# 03 - Reflection & Beyond

**Goal:** Consolidate what you've learned, understand common pitfalls, and know where to go next.

---

## Q&A: Key Questions From the Learning Journey

### Why SwinUNETR and not plain U-Net?

U-Net uses convolutions — each layer sees a small local area (3×3 kernel). To see the whole image, you need many layers stacked.

SwinUNETR uses **windowed attention** from Swin Transformer as the encoder. This lets the model see relationships across the whole volume more efficiently.

For spine segmentation this matters because:
- Vertebrae are **repetitive structures** — the model needs to understand the sequence (C1→C2→...→L5)
- Local context alone can't distinguish T8 from T9 — they look identical in isolation
- Attention lets each vertebra "look at" its neighbors to figure out its identity

Trade-off: SwinUNETR is slower to train and needs more memory, but produces better results on structured anatomy.

---

### Why a 2-stage pipeline (region → instance)?

A single model trying to label all 29 vertebrae in a full CT at 1mm resolution would need enormous GPU memory.

The 2-stage approach is a **coarse-to-fine** strategy:
1. **Region model (2mm):** Low resolution, fast. "Where is the thoracic region?"
2. **Instance model (1mm):** High resolution, focused. "Which specific vertebra is this?"

This is the same idea as sliding window inference — process manageable pieces, then combine.

---

### Why Dice + CrossEntropy loss?

In segmentation, **background dominates**. A spine CT is 99% non-spine. If you only use CrossEntropy, the model can get 99% accuracy by predicting "background everywhere."

**Dice loss** directly measures overlap between prediction and ground truth, ignoring the background. But Dice alone has noisy gradients.

**DiceCE = Dice + CrossEntropy** gives you:
- Dice: focus on the structures you care about
- CE: stable gradients for optimization

This is the standard for medical image segmentation.

---

### Why SmartCache?

3D medical volumes are huge. A single spine CT is ~512×512×300 voxels × 4 bytes = ~300 MB. Loading 100 CTs = 30 GB.

MONAI's **SmartCache** keeps a portion of data in RAM and dynamically replaces samples:
- Cache 64-96 samples
- Replace 25% each epoch with fresh samples from disk
- Background workers handle loading while GPU trains

This balances speed (cached data is instant) with diversity (fresh samples prevent overfitting).

---

### Why sliding window inference?

A full CT at 1mm resolution is ~512×512×300 voxels. The model expects 128×128×96 patches.

Sliding window:
1. Slide a 128×128×96 window across the volume
2. Run inference on each patch
3. Average overlapping predictions (80% overlap reduces edge artifacts)

The 4 `simultaneous_patch` setting means 4 patches are processed in one batch — a GPU memory trade-off.

---

### DVC vs Azure ML Data Assets?

**Use DVC when:**
- You want to stay cloud-agnostic
- Your team already uses Git
- You need offline access to data
- Free is important

**Use Azure Data Assets when:**
- You're fully committed to Azure
- You want a UI for non-technical team members
- You use Azure ML compute for training
- Built-in immutability matters (FDA)

**You can use both.** DVC for the data catalog (portable), Azure ML Data Assets for jobs running on Azure compute.

---

### MLflow vs TensorBoard?

They're **complementary**, not replacements:

| | TensorBoard | MLflow |
|--|------------|--------|
| Best at | Detailed training curves, gradient histograms | Comparing runs, finding best config |
| Stores | Logs (events files) | Params + metrics + artifacts |
| Queryable | No (visual only) | Yes (search_runs, SQL-like) |
| Model registry | No | Yes |

Keep TensorBoard for debugging training. Add MLflow for everything else.

## Common Pitfalls in Medical ML

### 1. Data Leakage

**The mistake:** Splitting by scan, not by patient. If a patient has 3 CT scans, putting 2 in training and 1 in validation means the model has "seen" that patient.

**The fix:** Always split at the **patient level**. All scans from one patient go to the same set.

```python
# WRONG: random split by scan
train_scans, val_scans = train_test_split(all_scans, test_size=0.2)

# RIGHT: split by patient, then expand to scans
train_patients, val_patients = train_test_split(all_patients, test_size=0.2)
train_scans = [s for s in all_scans if s.patient_id in train_patients]
val_scans = [s for s in all_scans if s.patient_id in val_patients]
```

---

### 2. Class Imbalance

In spine segmentation:
- Background: ~97% of voxels
- Vertebrae: ~3% of voxels
- Individual vertebra (e.g., C1): ~0.1% of voxels

**Solutions used in your pipeline:**
- DiceCE loss (handles imbalance)
- `RandCropByPosNegLabel` (crops around foreground)
- `foreground_ratio` in config (ensures patches contain structures)

---

### 3. Reproducibility

"I ran the same training and got different results."

**Causes:**
- Random seed not set
- Non-deterministic CUDA operations
- Different data order (shuffling)
- Different library versions

**Your pipeline's approach:**
```yaml
# In config
seed: 42
```

But this alone isn't enough. You also need:
- `torch.manual_seed(42)` + `torch.cuda.manual_seed_all(42)`
- `torch.backends.cudnn.deterministic = True` (slower but reproducible)
- Pinned library versions (requirements.txt)
- DVC-tracked data (exact same input)

---

### 4. 3D Memory Constraints

A 512×512×300 volume at float32 = 300 MB. After data augmentation, model activations, and gradients, one sample can consume 4-8 GB of GPU memory.

**Solutions:**
- Patch-based training (128×128×96 patches, not full volume)
- Mixed precision (`torch.cuda.amp`) — halves memory usage
- Gradient checkpointing (`use_checkpoint=True` in SwinUNETR)
- SmartCache (controls how much data is in RAM)
- Smaller batch sizes for higher resolution (batch=2 at 1mm vs batch=8 at 2mm)

---

### 5. Overfitting on Small Datasets

Medical datasets are typically small (50-500 patients). The model can memorize the training data.

**Your pipeline's defenses:**
- Data augmentation: rotation, scaling, intensity shifts
- Validation monitoring: check Dice every 500-1000 steps
- Save best model (not last): `best_metric_model.pth`

**Could improve:**
- Early stopping (stop training when validation stops improving)
- Cross-validation (train on multiple folds)
- Pre-trained weights (transfer learning from larger datasets)

## FDA 510(k) Considerations

### What is Software as Medical Device (SaMD)?

Your spine segmentation software is classified as **SaMD** — software that is itself a medical device (not software that controls a physical device).

FDA cares about:

| Requirement | What it means | What you need |
|-------------|--------------|---------------|
| **Software description** | What does it do? | Architecture docs (notebook 01-02 are a start) |
| **Intended use** | What clinical problem? | Spine vertebra identification |
| **Risk classification** | How dangerous if wrong? | Class II (moderate risk) |
| **Verification** | Does the code work correctly? | Unit tests, integration tests |
| **Validation** | Does it produce correct results? | Clinical validation on holdout data |
| **Data management** | How is data handled? | DVC versioning, data registry |
| **Change management** | How are updates tracked? | Git + MLflow + model registry |
| **Traceability** | Can you trace model → data → code? | DVC + MLflow lineage chain |

### What Auditors Actually Ask For

1. **"Show me the training data for this model."**
   → MLflow run params → DVC hash → exact dataset

2. **"What changed between model v8 and v9?"**
   → `dvc params diff` + `dvc metrics diff` + MLflow comparison

3. **"Can you reproduce the training?"**
   → `git checkout <commit>` + `dvc pull` + `dvc repro`

4. **"What's your test coverage?"**
   → pytest reports + validation metrics + holdout set results

5. **"How do you ensure the model doesn't degrade in production?"**
   → Monitoring plan (input validation, output confidence thresholds)

## Going Beyond: Tools and Concepts to Know

### nnU-Net

**What:** An auto-configuring segmentation framework by the DKFZ (German Cancer Research Center). Given a dataset, it automatically picks the best U-Net variant, preprocessing, augmentation, and training schedule.

**Why it matters:** It's the baseline everyone compares against in medical segmentation. If your model doesn't beat nnU-Net, there's usually a problem.

**For your team:** Could be used as a baseline to validate that SwinUNETR actually adds value over auto-configured U-Net.

---

### MONAI Label

**What:** An interactive annotation tool from the MONAI ecosystem. Integrates with 3D Slicer (a medical image viewer). The model suggests annotations, the user corrects, the model learns from corrections.

**Why it matters:** Annotating spine CTs is expensive (radiologist time). MONAI Label can 10x annotation speed through active learning.

**For your team:** If you need to expand your dataset, this is the tool to use instead of manual annotation.

---

### Foundation Models for Medical Imaging

**What:** Large pre-trained models that can be fine-tuned for specific tasks.

| Model | What it does |
|-------|--------------|
| SAM (Segment Anything) | General image segmentation, adaptable to medical |
| MedSAM | SAM fine-tuned on 1M+ medical images |
| BiomedCLIP | Vision-language model for medical images |
| STU-Net | Scalable U-Net pre-trained on 14,000 CT scans |

**Why it matters:** Instead of training from scratch, you fine-tune a foundation model. Needs less data, trains faster, often performs better.

**For your team:** Worth experimenting with as a pre-training strategy. Your SwinUNETR could be initialized with pre-trained weights instead of random initialization.

---

### ONNX Export

**What:** Open Neural Network Exchange — a standard format for ML models. Convert PyTorch → ONNX → run with ONNX Runtime.

**Why it matters:** ONNX Runtime is often 2-4x faster than native PyTorch for inference. Also enables running on different hardware (CPU optimized, TensorRT for NVIDIA GPUs).

```python
# Export
torch.onnx.export(model, dummy_input, "model.onnx")

# Run with ONNX Runtime
import onnxruntime
session = onnxruntime.InferenceSession("model.onnx")
result = session.run(None, {"input": input_array})
```

**For your team:** Could significantly speed up the inference pipeline (30-80s → 15-40s).

---

### TorchScript / libtorch

**What:** PyTorch's way to compile models for production C++ deployment.

**Why it matters:** Your production code already uses PyTorch in Python. TorchScript/libtorch lets you run the same model in C++ (faster, no Python overhead, easier to embed in medical devices).

**For your team:** A future optimization path if inference speed becomes critical.

---

### Federated Learning

**What:** Train a model across multiple hospitals without sharing patient data. Each hospital trains locally, only model updates are shared.

**Why it matters:** Medical data is sensitive (HIPAA, GDPR). Federated learning lets you train on more diverse data without data sharing agreements.

**For your team:** Relevant if you want to collaborate with hospitals. MONAI + NVFlare (NVIDIA) supports this.

## What You've Learned: The Full Map

```
Phase 0: CONCEPTS                Phase 1: PYTORCH
├── Neural networks              ├── Tensors & operations
├── Training & loss functions     ├── Autograd (backpropagation)
├── GPUs & parallelism           ├── nn.Module (building models)
└── CNNs (convolutions)          ├── Training loop pattern
                                 └── MNIST classifier (end-to-end)

Phase 2: SEGMENTATION            Phase 3: TRANSFORMERS
├── Segmentation concepts         ├── Attention mechanism (Q/K/V)
├── U-Net architecture            ├── Vision Transformer (ViT)
├── Training a U-Net              ├── Swin Transformer (windows)
└── MONAI framework               └── SwinUNETR (your production model)

Phase 4: MLOPS                   Phase 5: INTEGRATION
├── DVC (data versioning)         ├── Codebase analysis
├── MLflow (experiment tracking)  ├── Target architecture blueprint
├── Model Registry                └── Reflection & beyond (this notebook)
├── DVC Pipelines
├── DVC + MLflow together
└── Azure ML Studio guide
```

You now understand:
- **How** neural networks work (Phase 0-1)
- **How** segmentation models work (Phase 2)
- **Why** your production model is designed the way it is (Phase 3)
- **How** to version data and track experiments (Phase 4)
- **What** the target architecture should look like (Phase 5)

## Action Plan: What to Do Next

Prioritized by impact and effort:

| Priority | Action | Effort | Impact |
|----------|--------|--------|--------|
| **1** | Add DVC to training repo + track data | 1 day | High — instant lineage |
| **2** | Add MLflow logging to `training_semantic.py` | 1 day | High — experiment tracking |
| **3** | Set up Azure Blob as DVC remote | Half day | High — data off NAS |
| **4** | Create `maia-data-registry` repo | 1 day | Medium — catalog |
| **5** | Extract `maia-ml-core` shared package | 1 week | Medium — eliminates duplication |
| **6** | Define DVC pipeline (`dvc.yaml`) | 2 days | Medium — reproducibility |
| **7** | Set up MLflow server (or Azure ML) | 1-2 days | Medium — centralized tracking |
| **8** | Set up MLflow Model Registry | 1 day | Medium — model lifecycle |
| **9** | Update inference to pull from registry | 2-3 days | Medium — decouples build |
| **10** | Add unit tests to shared package | 1 week | Low-medium — quality |
| **11** | Document full lineage for FDA | 1 week | High — compliance |
| **12** | Team training on new workflow | 1 week | High — adoption |

**Start with actions 1-3.** They take 2-3 days and give you the biggest wins: data versioning, experiment tracking, and cloud storage. Everything else builds on top.

## Final Thought

You came in as an infra/devops/web engineer wanting to understand ML enough to architect the right solution.

You now understand:
- The ML fundamentals (not just buzzwords)
- Why the production code is designed the way it is
- What's missing (and it's mostly infrastructure, not ML)
- How to fix it (DVC + MLflow + proper repo structure)
- The migration path (progressive, not big-bang)

The ML team builds models. You build the infrastructure that makes those models traceable, reproducible, and FDA-ready. That's the value you bring.

**Learning plan complete.**